# 10 — Scenario Template

**Proves:** SDK usability — the 5-method contract is implementable from a template
**Module built:** `scenarios/_template/scenario.py`

A minimal 'coin flip with bias' scenario demonstrates the implementation checklist.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np

from scenarios._template.scenario import CoinFlipScenario
from sdk.core.engine import BranchedSimulationEngine, CounterfactualMode
from sdk.core.scenario import TimeConfig

## 1. The 5-Method Contract

Every scenario implements exactly 5 methods:
1. `create_population()` — initialize state
2. `step()` — domain physics (PURE)
3. `predict()` — ML model simulation
4. `intervene()` — policy application
5. `measure()` — outcome recording

In [ ]:
tc = TimeConfig(
    n_timesteps=20,
    timestep_duration=1/52,
    timestep_unit='week',
    prediction_schedule=[5, 10, 15],
)

scenario_b = CoinFlipScenario(time_config=tc, seed=42)
engine_b = BranchedSimulationEngine(scenario_b, CounterfactualMode.BRANCHED)
results_b = engine_b.run(100)

# Factual vs counterfactual totals
total_factual = sum(
    results_b.outcomes[t].events.sum() for t in results_b.outcomes
)
total_cf = sum(
    results_b.counterfactual_outcomes[t].events.sum()
    for t in results_b.counterfactual_outcomes
)

print(f"Unit of analysis: {results_b.unit_of_analysis}")
print(f"Mode: {results_b.counterfactual_mode}")
print(f"Timesteps: {tc.n_timesteps}")
print(f"Prediction schedule: {tc.prediction_schedule}")
print(f"Entities: 100")
print(f"")
print(f"Total factual flips (heads):        {total_factual:.0f}")
print(f"Total counterfactual flips (heads):  {total_cf:.0f}")
print(f"Difference (intervention effect):    {total_cf - total_factual:.0f}")

## 2. Purity Test

In [ ]:
# Run NONE mode with same seed
scenario_n = CoinFlipScenario(time_config=tc, seed=42)
engine_n = BranchedSimulationEngine(scenario_n, CounterfactualMode.NONE)
results_n = engine_n.run(100)

# Compare factual outcomes element-wise at every timestep
for t in range(tc.n_timesteps):
    if t in results_b.outcomes and t in results_n.outcomes:
        diff = np.abs(
            results_b.outcomes[t].events - results_n.outcomes[t].events
        ).max()
        assert diff == 0.0, f"PURITY VIOLATION at t={t}: max diff = {diff}"

print("PURITY PASSED: BRANCHED factual is identical to NONE at every timestep.")
print("The coin-flip template satisfies the step purity contract.")

## Key Insights

1. A new scenario requires implementing 5 methods and declaring `unit_of_analysis`.
2. Use the correct RNG stream in each method (see checklist in `scenarios/_template/scenario.py`).
3. Always run the BRANCHED-vs-NONE purity test on your scenario.

The SDK now has 3 validated scenarios: stroke prevention, no-show overbooking, and the coin-flip template.